<left><img width=25% src="img/gw_monogram_2c.png" alt="GW mark"></left>

# Lecture 1: Machine Learning for Electrical Engineers

### ECE6210 Machine Intelligence, Fall 2026

Armin Mehrabian  
The George Washington University

# Course delivery

- Lectures are online.
- The midterm examination is in person on October 21.
- The final examination is in person during December 11 to 17.
- The exact final examination date will be assigned later.

# Meeting time

The class meets on Wednesday from 9:30 a.m. to 11:30 a.m. in Washington, DC.

- August 26 through October 28: 5:30 p.m. to 7:30 p.m. in Baku
- November 4 through December 2: 6:30 p.m. to 8:30 p.m. in Baku

There is no class on November 25.

# Assessment

| Assessment | Weight | Format |
|---|---:|---|
| Midterm examination | 40% | In person |
| Final examination | 40% | In person |
| Group project | 20% | Two students per group |

Short notebook exercises will support examination preparation.

# Required background

Students should have basic knowledge of:

- Programming in Python, MATLAB, C, or C++
- Matrix operations and eigenvectors
- Probability distributions and Bayes' rule
- Signals and systems

Prior machine learning study is not required.

# Course scope

The course studies machine learning methods and their engineering use.

Hardware is considered as an implementation constraint. We will discuss computation, memory, latency, energy, and numerical precision.

The course does not require VLSI, HDL, or processor design.

# Learning outcomes

After this course, students should be able to:

1. Formulate an engineering problem as a learning problem.
2. Select a model, objective function, and optimizer.
3. Evaluate generalization using suitable data and metrics.
4. Interpret the effect of noise and limited data.
5. Compare accuracy, memory, latency, and interpretability.

# Course sequence

1. Data and learning problems
2. Linear regression and classification
3. Generalization and probabilistic models
4. PCA and clustering
5. Neural networks and one-dimensional convolution
6. Trees, ensembles, and engineering model comparison

# Group project

The project applies machine learning to an engineering problem.

Suitable areas include:

- Signals and sensing
- Communications
- Control and system monitoring
- Energy systems
- Biomedical measurements
- Fault detection

Hardware implementation is optional.

<left><img width=25% src="img/gw_monogram_2c.png" alt="GW mark"></left>

# Part 1: Learning from engineering measurements

# An engineering measurement

A measurement can be represented as

$$x = s + n,$$

where $s$ is the quantity of interest and $n$ represents noise and unmodeled effects.

A learning method uses measured examples to estimate a useful relationship.

# Example: sensor calibration

Consider a temperature sensor. Its voltage depends on temperature.

$$v = aT + b + n$$

The constants $a$ and $b$ are not known exactly. Measurements can be used to estimate the relation between voltage and temperature.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(6210)
temperature = np.linspace(0, 100, 25)
voltage = 0.40 + 0.025 * temperature
voltage += rng.normal(0, 0.06, size=temperature.size)

X = np.column_stack([np.ones_like(voltage), voltage])
theta = np.linalg.lstsq(X, temperature, rcond=None)[0]
temperature_hat = X @ theta
rmse = np.sqrt(np.mean((temperature_hat - temperature) ** 2))

print(f"Estimated model: T = {theta[0]:.2f} + {theta[1]:.2f} v")
print(f"Calibration RMSE: {rmse:.2f} degrees C")

In [ ]:
order = np.argsort(voltage)
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.scatter(voltage, temperature, label="Measurements")
ax.plot(voltage[order], temperature_hat[order], "k-", label="Fit")
ax.set(xlabel="Sensor voltage, V", ylabel="Temperature, degrees C")
ax.set_title("Sensor calibration")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

# Interpretation

The fitted model estimates temperature from a voltage measurement.

The error is not zero because the measurements contain noise. The fitted relation must also be tested using measurements that were not used for fitting.

### Question

Would the same calibration remain valid after the sensor ages or the operating environment changes?

# Machine learning

Machine learning studies methods that estimate patterns or relationships from data.

A trained model should perform well on new data from the intended operating conditions.

# Main components

A learning problem contains:

$$\text{data} + \text{model class} + \text{objective} + \text{optimizer}.$$

These components produce a fitted model. The fitted model is then evaluated on new data.

# Supervised learning

A supervised dataset contains input and target pairs:

$$\mathcal{D} = \{(x^{(i)}, y^{(i)})\}_{i=1}^{n}.$$

The model estimates $y$ from $x$.

Supervised learning includes:

- Regression for a continuous target
- Classification for a discrete target

# Supervised EE examples

- Estimate temperature from sensor voltage.
- Detect a fault from vibration measurements.
- Classify a modulation type from received samples.
- Estimate energy demand from historical measurements.
- Predict a system response from an input signal.

# Unsupervised learning

An unsupervised dataset contains inputs without target labels:

$$\mathcal{D} = \{x^{(i)}\}_{i=1}^{n}.$$

The objective is to identify useful structure in the measurements.

# Unsupervised EE examples

- Identify operating conditions from sensor data.
- Reduce the dimension of correlated measurements.
- Detect unusual system behavior.
- Separate signal structure from measurement noise.

# Reinforcement learning

Reinforcement learning studies sequential decisions based on actions and rewards.

Examples include control, resource allocation, and autonomous systems.

This course introduces the concept but does not develop reinforcement learning algorithms.

<left><img width=25% src="img/gw_monogram_2c.png" alt="GW mark"></left>

# Part 2: Formulating a learning problem

# Data

Data are measured or simulated observations.

Important questions include:

- What quantity was measured?
- How was it sampled?
- What units are used?
- What noise and bias are present?
- Do the data represent the intended operating conditions?

# Model class

A model maps an input $x$ to an output:

$$\hat{y} = f_{\theta}(x).$$

The parameters $\theta$ are estimated from data. The model class defines the possible forms of $f_{\theta}$.

# Objective function

The objective function measures the quality of a parameter choice.

For regression, one possible objective is mean squared error:

$$J(\theta) = \frac{1}{n}\sum_{i=1}^{n}\left(y^{(i)} - f_{\theta}(x^{(i)})\right)^2.$$

# Optimizer

The optimizer selects parameters that reduce the objective function:

$$\theta^* = \arg\min_{\theta} J(\theta).$$

Different optimizers can have different computation and memory requirements.

# Generalization

A small training error is not sufficient.

The model should also have a small error on new data from the intended operating conditions. This property is called generalization.

Training, validation, and test data have different purposes. We will define these data partitions in Lecture 2.

<left><img width=25% src="img/gw_monogram_2c.png" alt="GW mark"></left>

# Part 3: Hardware-aware machine learning

# A model operates on a physical system

A practical system may include:

$$\text{sensor} \rightarrow \text{sampling} \rightarrow \text{features} \rightarrow \text{model} \rightarrow \text{decision}.$$

Each stage can introduce delay, noise, limited precision, and data loss.

# Engineering requirements

A model may need to satisfy several requirements:

- Prediction accuracy
- Maximum latency
- Available memory
- Energy or power limit
- Numerical precision
- Reliability under changing conditions

# Memory example

A model contains 250,000 parameters. Storage depends on the numerical representation.

In [ ]:
parameter_count = 250_000
representations = {"float32": 4, "float16": 2, "int8": 1}

for name, bytes_per_parameter in representations.items():
    memory_kib = parameter_count * bytes_per_parameter / 1024
    print(f"{name:7s}: {memory_kib:7.1f} KiB")

# Numerical precision

Lower precision can reduce memory and computation. It can also change the model output.

The acceptable precision depends on the model, data, and engineering requirement. This must be evaluated using measurements.

# Hardware questions in this course

We will periodically ask:

- Which operations dominate computation?
- How much memory does the model require?
- Can dimensionality reduction reduce data movement?
- Can lower precision be used without unacceptable error?
- Does the model meet the required latency?

# Responsible engineering use

A model should be evaluated beyond average accuracy.

- Document the data source and operating conditions.
- Report uncertainty and important failure cases.
- Check whether performance changes across relevant groups or conditions.
- Maintain a non-learning baseline when possible.
- Define when human review is required.

# Short activity

Select one EE application. State the following items:

1. Input measurement
2. Desired output
3. Main source of noise or uncertainty
4. One performance metric
5. One hardware or system constraint

# Summary

- Machine learning estimates useful relationships from data.
- A learning problem contains data, a model class, an objective, and an optimizer.
- Generalization must be evaluated on new data.
- Engineering use requires attention to noise and operating conditions.
- Computation, memory, latency, and precision can affect model selection.

# Next lecture

Lecture 2 defines datasets, features, targets, model classes, objective functions, and optimizers.

We will also study data splitting, normalization, and leakage.